# Plant Disease Detection - Fast GDrive Cache Pipeline
This notebook automatically manages your dataset. It will zip and save the dataset to your Google Drive the first time you run it. On future runs, it will instantly unpack the zip from your Drive instead of re-downloading.

In [ ]:
import os
import shutil
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from google.colab import drive

try:
    import kagglehub
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "kagglehub"])
    import kagglehub

# 1. Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# 2. Intelligent Dataset Downloading & Caching
DATASET_ID = 'vipoooool/new-plant-diseases-dataset'
drive_zip_path = '/content/drive/MyDrive/plant_dataset.zip'
local_zip_path = '/content/plant_dataset.zip'
local_extract_path = '/content/dataset'

if os.path.exists(drive_zip_path):
    print("[INFO] Found zipped dataset in Google Drive. Copying to local storage...")
    shutil.copy(drive_zip_path, local_zip_path)
    print("[INFO] Unzipping dataset locally for high-speed training...")
    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall(local_extract_path)
    base_dir = local_extract_path
else:
    print("[INFO] Zipped dataset not found in Drive. Downloading via kagglehub...")
    dataset_path = kagglehub.dataset_download(DATASET_ID)
    
    print("[INFO] Zipping dataset for future use... this might take a minute.")
    shutil.make_archive('/content/plant_dataset', 'zip', dataset_path)
    
    print("[INFO] Transferring zip file to Google Drive...")
    shutil.copy('/content/plant_dataset.zip', drive_zip_path)
    print(f"[INFO] Success! Dataset saved to Drive at: {drive_zip_path}")
    base_dir = dataset_path

# Dynamically find the train and valid directories (handles nested folders safely)
for root, dirs, files in os.walk(base_dir):
    if 'train' in dirs and 'valid' in dirs:
        base_dir = root
        break

train_dir = os.path.join(base_dir, 'train')
valid_dir = os.path.join(base_dir, 'valid')
print(f"[INFO] Train directory set to: {train_dir}")
print(f"[INFO] Validation directory set to: {valid_dir}")

In [ ]:
# 3. Dataloaders & Transforms
BATCH_SIZE = 32
IMG_SIZE = 224

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'valid': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {
    'train': datasets.ImageFolder(train_dir, data_transforms['train']),
    'valid': datasets.ImageFolder(valid_dir, data_transforms['valid'])
}

dataloaders = {
    'train': DataLoader(image_datasets['train'], batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True),
    'valid': DataLoader(image_datasets['valid'], batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
}

class_names = image_datasets['train'].classes
print(f"[INFO] Successfully loaded {len(class_names)} disease classes.")

In [ ]:
# 4. Model Architecture Setup
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {device}")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# 5. Training Loop
num_epochs = 5

for epoch in range(num_epochs):
    print()
    print(f'Epoch {epoch+1}/{num_epochs}')
    print('-' * 15)

    for phase in ['train', 'valid']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)

                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = running_corrects.double() / len(image_datasets[phase])

        print(f'{phase.capitalize()} Phase -> Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}')

print("\n[INFO] Training complete!")